# Guide to getting started authoring your own protocol registry YAML

For: The SCOOP project

Last updated: 2026/08/04 by Brianna Argall

First, take a rough look at an existing profile, such as nsclc, to get an idea of what the final YAML can look like. 

When you are writing the YAML, it is highly recommended that you set up a notebook and that you use this while you are writing your YAML protocol registry. Give the notebook a descriptive name and write notes for yourself in markdown cells to help you remember the purpose of your notebook.

## Get required imports working:

In the example notebooks to help you, the first block of code is something like:

In [1]:
import os
import sys
from pathlib import Path


# Here we define the function that lets SCOOP be found from within the SCOOP
# folder.  This allows navigation to files inside SCOOP to work.
# It defines the repo (code repository) root as containing a notebooks and scoop folder
# If you not working inside the SCOOP repository it will not work - which is
# fine as long as you can import SCOOP - only use this function if relevant to you.
def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "scoop").exists() and (candidate / "notebooks").exists():
            return candidate
    raise RuntimeError("Could not locate the SCOOP repo root.")

# Here we call on the function to make sure our code can find SCOOP
ROOT = find_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

CONFIG_ROOT = ROOT / "scoop" / "configuration"

Whether you are working in the notebooks folder inside SCOOP or not, make sure you can import from scoop / a file within SCOOP. 

In [2]:
from scoop.utils.treatment_selection import ConditionDevTools, show_conditions

In order to investigate what's available in the HemOncKB you will need access to the HemOncKB. This project expects access to data models from hemonc_import to understand the data requirements and relationships in the HemOncKB. hemonc_import is from the Australian Cancer Data Network (not HemOnc). If you do not have this and need it for some code to run you will get an error message telling you to install it.

For interactions with the database, you will need to pass a session to the relevant function. To do this, you can specify in your environment (using the .env file) the database engine url.

In [3]:
import sqlalchemy as sa
import sqlalchemy.orm as so

HEMONC_ENGINE_URL = os.environ["HEMONC_ENGINE_URL"]

def _require_engine_url(engine_url: str | None = None) -> str:
    resolved = engine_url or HEMONC_ENGINE_URL
    if not resolved:
        raise ValueError("Set HEMONC_ENGINE_URL or pass engine_url explicitly.")
    return resolved

engine = sa.create_engine(_require_engine_url(HEMONC_ENGINE_URL))

You should now have the engine which you can use to get session access. Let's start by looking up what conditions are available in your copy of the HemOncKB

In [4]:
with so.Session(engine) as session:
    display(show_conditions("lung", session)) # try changing "lung" 

,condition,condition_cui
0,ALK-positive Non-small cell lung cancer,30824
1,BRAF-mutated Non-small cell lung cancer,30898
2,c-Met-positive Non-small cell lung cancer,154361
3,EGFR-mutated Non-small cell lung cancer,30910
4,HER2-mutated Non-small cell lung cancer,52958
5,KRAS-mutated Non-small cell lung cancer,54044
6,Lung cancer,46104
7,MET-mutated Non-small cell lung cancer,13094
8,Non-small cell lung cancer,642
9,Non-small cell lung cancer nonsquamous,25309


SCOOP accepts conditions lists as either lists of strings or as lists of CUIs (as integers).
Spelling must be exact. Behind the scenes, strings are converted to CUIs by looking them up in the database. Therefore, sometimes if you choose strings rather than CUIs session must be passed.

## Create a helper with ConditionDevTools

Let's make a helper to see what components / component classes are available. When you specify a component in SCOOP it will check across relevant columns and return a match in any of those columns as a hit for that component/row.

Let's check which columns are specified as relevant:

In [5]:
from scoop.utils.hemonc_helpers import COMPONENT_STR_COLS

print(COMPONENT_STR_COLS)

['canmed_major_class', 'canmed_minor_class', 'main_class', 'component']


We now want to check what components (and their classes) are available for the conditions we are interested in. Change the conditions list to the conditions you want to look at, using the show conditions tool from earlier to help you find these. To help us do this, we'll make a helper.

In [6]:
MY_CONDITIONS = ["Lung cancer", "Non-small cell lung cancer"]

with so.Session(engine) as session:
    my_helper = ConditionDevTools(conditions=MY_CONDITIONS, session=session)

Our helper will only return results relevant to the conditions list used when making it.
We can use it to search for relevant components like how we searched for conditions before:

In [7]:
with so.Session(engine) as session:
    my_helper.show_component_with_substring(session, "met") # Try other searches

['Anti-c-Met antibody-drug conjugate', 'Antimetabolite', 'Dexamethasone', 'Methotrexate', 'Metoclopramide', 'Pemetrexed', 'Trametinib']


But we can also produce a dataframe of all of our relevant components which we can explore like we would any other pandas dataframe. This may take a bit the first time, but if you ask for the same dataframe the helper will remember the first version and give that to you quickly.

In [8]:
with so.Session(engine) as session:
    my_components = my_helper.components.get_dataframe(session)

In [9]:
# id here refers to sig id. This is created by hemonc_import, not provided with HemOncKB
# it can be helpful for you to look up sigs for more info but do not
# expect it to be consistent across different versions
my_components.head()

,id,regimen,variant_cui,component,component_cui,drug_inn,main_class,canmed_major_class,canmed_minor_class,study,phase
0,3428.0,Carboplatin and Paclitaxel (CP),130386.0,Carboplatin,88.0,carboplatin,Platinum agent,Alkylating agent,Platinum agent,A4021016|A8501001|ATTRACT-1|ECOG E4599|IPASS|K...,NaN
1,3429.0,Carboplatin and Paclitaxel (CP),130386.0,Paclitaxel,379.0,paclitaxel,Taxane,Microtubule inhibitor,Taxane,A4021016|A8501001|ATTRACT-1|ECOG E4599|IPASS|K...,NaN
2,19414.0,Osimertinib monotherapy,133383.0,Osimertinib,375.0,osimertinib,EGFR inhibitor,Tyrosine kinase inhibitor,EGFR inhibitor,ADAURA,Adjuvant
3,6183.0,Cisplatin and Vinorelbine (CVb),141836.0,Cisplatin,105.0,cisplatin,Platinum agent,Alkylating agent,Platinum agent,ADJUVANT/CTONG1104|ALINA,NaN
4,6184.0,Cisplatin and Vinorelbine (CVb),141836.0,Vinorelbine,540.0,vinorelbine,Vinca alkaloid,Microtubule inhibitor,Vinca alkaloid,ADJUVANT/CTONG1104|ALINA,NaN


In [10]:
my_components.info()

<class 'pandas.DataFrame'>
Index: 782 entries, 0 to 782
Data columns (total 11 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   id                  782 non-null    float64
 1   regimen             782 non-null    str    
 2   variant_cui         733 non-null    float64
 3   component           782 non-null    str    
 4   component_cui       782 non-null    float64
 5   drug_inn            724 non-null    str    
 6   main_class          743 non-null    str    
 7   canmed_major_class  700 non-null    str    
 8   canmed_minor_class  669 non-null    str    
 9   study               782 non-null    str    
 10  phase               189 non-null    str    
dtypes: float64(3), str(8)
memory usage: 174.9 KB


## Check broad category exists for each component

For non-small cell lung cancer I made sure every component maps to a broad category. One day I hope to make this mapping automatic - for now it is manually specified in class maps. If you are looking at another condition, one of your recommended first tasks is to check if there are any components for which broad category is not specifed. This is an example workflow 

In [11]:
with so.Session(engine) as session:
    my_categorised = my_helper.components.get_dataframe(session, categorise=True)

First we check which broad categories are represented for the components relevant to you:

In [12]:
my_categorised["broad_category"].value_counts(dropna = False)

broad_category
chemotherapy     616
immunotherapy     83
supportive        51
radiotherapy      32
Name: count, dtype: int64

If you see an NA category above, you will want to make sure all of those uncategorised components get a category mapping.

Note that these classifications are per component and do not consider context. For example, leucovorin is always classified as "supportive" currently.

In [13]:
my_categorised[my_categorised["broad_category"].isna()]

,id,regimen,variant_cui,component,component_cui,drug_inn,main_class,canmed_major_class,canmed_minor_class,study,phase,broad_category


My recommendation is that you specify using the bigger class groups where you can.
The class maps live in hemonc_helpers, where you can easily edit them.

If you add a new broad category e.g. endocrine, you will also need to edit protocol_registry.py so we know how to treat your new category. Seach for BroadCategory and edit where appropriate.

For example: 

`BroadCategory = Literal["chemotherapy", "radiotherapy", "immunotherapy", "surgery", "supportive",]` 

would become

`BroadCategory = Literal["chemotherapy", "radiotherapy", "immunotherapy", "surgery", "supportive", "endocrine"]`

and 

```
DEFAULT_CATEGORY_RESTRICTIONS: Final[Dict[BroadCategory, CategoryRestriction]] = {
    "chemotherapy":  {"type": "exact", "amount": 0},
    "radiotherapy":  {"type": "exact", "amount": 0},
    "immunotherapy": {"type": "exact", "amount": 0},
    "surgery":       {"type": "exact", "amount": 0},
    # NOTE: "supportive" intentionally excluded
}
```

would become:

```
DEFAULT_CATEGORY_RESTRICTIONS: Final[Dict[BroadCategory, CategoryRestriction]] = {
    "chemotherapy":  {"type": "exact", "amount": 0},
    "radiotherapy":  {"type": "exact", "amount": 0},
    "immunotherapy": {"type": "exact", "amount": 0},
    "surgery":       {"type": "exact", "amount": 0},
    "endocrine":     {"type": "exact", "amount": 0},
    # NOTE: "supportive" intentionally excluded
}
```

so that unless someone asks for an endocrine-including regimen, we assume that the presence of endocrine components excludes that regimen.

After you have updated the class maps in hemonc_helpers.py, restart your notebook so that the new version is imported and rerun the checks for NA components to see your progress. 

## Creating a protocol registry

Now that you know how to:
- find conditions you can use
- find relevant components for your condition group
- categorise these components if needed

You are ready to start building your protocol registry.

You need to be able to import and parse your YAML, for example, like this:

In [14]:
import yaml

REGISTRY_PATH = ROOT / "scoop" / "configuration" / "profiles" / "nsclc" / "nsclc_protocol_registry.yaml"
with REGISTRY_PATH.open("r", encoding="utf-8") as handle:
    raw = yaml.safe_load(handle)

The output `raw` is a dictionary and can be interacted with like other python dictionaries. The indentation level on the yaml determines the structure, so it's important to get this right. For example, every source phase entry should be idented to one level more than the text `source_phases` is.

```
nsclc:
    version: ...
    conditions: ...
    source_phases:
        example_source_phase: 
        ...
    phase_combinations:
        ...
    protocols:
        ...
    entries:
        ...
```

In [15]:
raw["nsclc"].keys()

dict_keys(['version', 'conditions', 'source_phases', 'phase_combinations', 'protocols', 'entries'])

In [16]:
raw["nsclc"]["source_phases"].keys()

dict_keys(['surgery_only', 'port', 'adj_osimertinib', 'neoadjuvant_chemo_ici', 'adjuvant_ici', 'adjuvant_pembrolizumab', 'adjuvant_atezolizumab_pembrolizumab', 'adjuvant_osimertinib', 'adjuvant_cisplatin_based', 'adjuvant_optchemo_alectinib', 'conc_crt', 'seq_crt_chemo', 'definitive_rt', 'consolidation_durvalumab', 'consolidation_osimertinib', 'egfr_intervention', 'alk_intervention', 'ros1_intervention', 'braf_intervention', 'ret_intervention', 'erbb2_intervention', 'ntrk_intervention', 'met_tki', 'met_ptdb', 'met_ptdb_w_ici', 'best_supportive_care', 'platinum_doublet_3rd_gen_agent', 'platinum_doublet_nonsquamous', 'ici_w_chemo_nonsquamous', 'ici_w_chemo_squamous', 'ici_monotherapy', 'ptdb_squamous', 'single_chemo_squamous', 'ptdb_nonsquamous', 'single_chemo_nonsquamous', 'atezolizumab'])

### First source phase example

Let's now look at an example of what a source phase looks like in the YAML:

```
    adjuvant_pembrolizumab:
        component_requirements:
            component_lists: "pembrolizumab" # autocorrected to [["pembrolizumab"]] in processing
            category_restrictions:
                immunotherapy: 
                    type: exact
                    amount: 1
        type: adjuvant
```

Here we can see an example of auto-rescue behaviour by SCOOP - if just `"pembrolizumab"` is provided SCOOP will rescue this into the required form for processing as `[["pembrolizumab"]]`. Not all deviations from the standard can be rescued - for example, misspellings. We can see the rescued form below:

In [17]:
from scoop.configuration import protocol_registry

with so.Session(engine) as session:
    nsclc_registry = protocol_registry.RegistryYaml.from_yaml_path(REGISTRY_PATH, session)

In [18]:
nsclc_registry.source_phases["adjuvant_pembrolizumab"]

SourcePhaseYaml(source_phase_id='adjuvant_pembrolizumab', condition_cuis=[46104, 642], component_requirements=ComponentRequirements(component_lists=[['pembrolizumab']], category_restrictions={'chemotherapy': CategoryRestriction(type='exact', amount=0), 'radiotherapy': CategoryRestriction(type='exact', amount=0), 'immunotherapy': CategoryRestriction(type='exact', amount=1), 'surgery': CategoryRestriction(type='exact', amount=0)}), study_restriction=None, curative_intent=True, regimens=None, type='adjuvant', standalone=True)

You will also notice that the conditions strings have been converted to CUIs and aside from supportive, it is assumed that any unspecified broad category should not be present in the returned sigs.

In [19]:
nsclc_registry.source_phases["adjuvant_pembrolizumab"].component_requirements

ComponentRequirements(component_lists=[['pembrolizumab']], category_restrictions={'chemotherapy': CategoryRestriction(type='exact', amount=0), 'radiotherapy': CategoryRestriction(type='exact', amount=0), 'immunotherapy': CategoryRestriction(type='exact', amount=1), 'surgery': CategoryRestriction(type='exact', amount=0)})


**Type** corresponds to the HemOncKB value of sig.phase. Capitalisation does not matter for the purpose of matching.
There is no instruction for "include all" - if you do not specify, it will be assumed you want not just any sig but a sig with NULL phase.

That being said, type is only as strict as it can be while still returning sigs. First, SCOOP will try to find sigs with your specified type, if it can't find any, it will go for sigs with NULL sig phase. If it can't find sigs with NULL phase, it will avoid subsetting on type.

If you do not like this behaviour, change `subset_df_on_phase()` in `treatment_selection.py`; this function is written using pandas functions rather than going through sqlalchemy and therefore should be relatively beginner-friendly to customise.


Let's see what's returned for adjuvant_pembrolizumab:

In [20]:
from scoop.utils.treatment_containers import ProtocolGenerator

# ignoring concurrent phase combinations for now
# but in general you are recommend to provide phase_unions if in your registry
protocol_generator = ProtocolGenerator(source_phases=nsclc_registry.source_phases, phase_unions={}) 

In [21]:
with so.Session(engine) as session:
    adj_pembro = protocol_generator.get_protocol_from_source_phase("adjuvant_pembrolizumab", session)

In [22]:
# the above returns a list so we take the first/0th item
adj_pembro[0]

Treatment Protocol: adjuvant_pembrolizumab (1 phase(s))

As you can see, the phases are all none, indicating that sigs with phase specified as adjuvant were not found.

Here, we retrieved monotherapies by 
- a) requiring the monotherapy component 
- b) restricting to allow only one item from that broad category only 
- c) not adding other broad categories. 

On its own, specifying a component only means that component must be present - it does not exclude anything which is not that component. 
For example, you could validly specify cisplatin in the component list - not specify any other components - and require exactly 2 different chemotherapy components.

### A more complicated source phase example

```
    ici_w_chemo_nonsquamous:
        conditions: "Non-small cell lung cancer squamous"  # [25309]
        component_requirements:
            component_lists: [["Platinum agent"], ["Immunostimulatory monoclonal antibody"]]
        type: "None"
```

Here, conditions information is provided which overrides the default conditions. This facilitates using more specific conditions that you do not want drawn on by all of the source phases.

We also specify the component list as requiring both a platinum agent AND an ICI. Component lists work like this: 

`[ [thing1 OR thing2 OR thing3...] AND [thinga OR thingb OR thingc...] AND [thingi OR thingj OR thingk ...] ...]`

There would be many valid combinations including: thing1 and thingb and thingi, or thing3 and thingc and thingk


We could obtain the same results* for ici_chemo_nonsquamous by using the component list:

`[["Carboplatin", "Cisplatin"], ["Immunostimulatory monoclonal antibody"]]`

\* carboplatin and cisplatin are not the only platinum agents in the HemOncKB but in this case results are the same

Since there are no restrictions on broad category at all, as specified here it would be possible for a concurrent CRT regimen which met the component requirements to be included. A more robust specification would require min 1 chemo and min 1 immuno. This would then automatically kick out any radiotherapy including regimen variants. However, if you are obtaining no results for a source phase, it is useful to temporarily disable the broad category restrictions to check that you haven't created mutually exclusive requirements. (Like asking for an ICI component and forgetting to allow any immunotherapies.)

In [23]:
# Let's see what we get:
with so.Session(engine) as session:
    ici_w_chemo_nsq = protocol_generator.get_protocol_from_source_phase("ici_w_chemo_nonsquamous", session)

In [24]:
ici_w_chemo_nsq[0]

Treatment Protocol: ici_w_chemo_nonsquamous (1 phase(s))

### More complicated: phase_unions

If you need logic that is more complicated than allowed by the above rules, you can combine phases together.
A sigbundle valid for any of the specified source phases is then valid for the phase union. 

In [25]:
# We'll now include the data on phase unions in our generator
better_generator = ProtocolGenerator(source_phases=nsclc_registry.source_phases, phase_unions=nsclc_registry.union_phases)

In [26]:
better_generator.phase_unions

{'met_intervention': PhaseUnionYaml(phase_id='met_intervention', source_phase_ids=['met_tki', 'met_ptdb', 'met_ptdb_w_ici']),
 'ici_monotherapy_or_ici_with_chemotherapy_squamous': PhaseUnionYaml(phase_id='ici_monotherapy_or_ici_with_chemotherapy_squamous', source_phase_ids=['ici_monotherapy', 'ici_w_chemo_squamous']),
 'ici_monotherapy_or_ici_with_chemotherapy_nonsquamous': PhaseUnionYaml(phase_id='ici_monotherapy_or_ici_with_chemotherapy_nonsquamous', source_phase_ids=['ici_monotherapy', 'ici_w_chemo_nonsquamous']),
 'chemotherapy_or_atezolizumab_squamous': PhaseUnionYaml(phase_id='chemotherapy_or_atezolizumab_squamous', source_phase_ids=['ptdb_squamous', 'single_chemo_squamous', 'atezolizumab']),
 'chemotherapy_or_atezolizumab_nonsquamous': PhaseUnionYaml(phase_id='chemotherapy_or_atezolizumab_nonsquamous', source_phase_ids=['ptdb_nonsquamous', 'single_chemo_nonsquamous', 'atezolizumab'])}

In [27]:
# Let's see what we get:
with so.Session(engine) as session:
    ici_monotherapy_or_ici_w_chemo_nsq = better_generator.get_protocol_from_phase_union(
        "ici_monotherapy_or_ici_with_chemotherapy_nonsquamous", session
        )

In [28]:
ici_monotherapy_or_ici_w_chemo_nsq[0]

Treatment Protocol: ici_monotherapy_or_ici_with_chemotherapy_nonsquamous (1 phase(s))

### Sequencing phases into protocols

Phases (direct source phases or phase unions) can be combined sequentially to form protocols. The order in which the phases are listed in the protocol YAML will be the order of the phases in the protocol. Protocols cannot currently reference other protocols - just source phases or phase unions. 

There is a special, optional rule which can be activated to require a shared component across phases of a protocol. This is useful for perioperative protocols. Additionally, to cater for cases such as this where only some instances of a phase are valid depending on another within protocol phase, the output from a protocol YAML is a list of protocol objects. This is most easily understood through an example:

In [29]:
nsclc_registry.protocols["conc_crt_followed_by_durvalumab"]

ProtocolYaml(protocol_id='conc_crt_followed_by_durvalumab', source_phase_ids=['conc_crt', 'consolidation_durvalumab'], optional_phases=None)

In [30]:
# Simple first
with so.Session(engine) as session:
    conc_crt_followed_by_durvalumab = better_generator.get_protocols_from_protocol_yaml(
        nsclc_registry.protocols["conc_crt_followed_by_durvalumab"], session
        )

In [31]:
# A list of protocols was returned
conc_crt_followed_by_durvalumab

[Treatment Protocol: conc_crt_followed_by_durvalumab (2 phase(s))]

In [32]:
# Let's look at the only protocol in the list
conc_crt_followed_by_durvalumab[0]

Treatment Protocol: conc_crt_followed_by_durvalumab (2 phase(s))

Multiple protocols are returned when the contents of one phase influence what's valid in another phase.

In [33]:
with so.Session(engine) as session:
    neoadj_or_peri = better_generator.get_protocols_from_protocol_yaml(
        nsclc_registry.protocols["neoadjuvant_or_perioperative_ici_w_chemo"], session
        )

In [34]:
neoadj_or_peri

[Treatment Protocol: neoadjuvant_or_perioperative_ici_w_chemo (3 phase(s)),
 Treatment Protocol: neoadjuvant_or_perioperative_ici_w_chemo (2 phase(s))]

In [35]:
neoadj_or_peri[0]

Treatment Protocol: neoadjuvant_or_perioperative_ici_w_chemo (3 phase(s))

In [36]:
neoadj_or_peri[1]

Treatment Protocol: neoadjuvant_or_perioperative_ici_w_chemo (2 phase(s))